# Setup

In [13]:
%%bash
/home/airflow/.venv/bin/python ./generate_data.py
/home/airflow/.venv/bin/python ./run_ddl.py

2026-03-24 15:09:39,065 - __main__ - INFO - Parsed command line arguments
2026-03-24 15:09:39,065 - __main__ - INFO - Starting TPC-H dataset generation pipeline
2026-03-24 15:09:39,065 - __main__ - INFO - Starting TPC-H table creation with scaling factor 0.1
2026-03-24 15:09:40,687 - __main__ - INFO - TPC-H tables created with scaling factor 0.1
2026-03-24 15:09:40,687 - __main__ - INFO - Starting export of TPC-H tables to csv format in ./data
2026-03-24 15:09:40,698 - __main__ - INFO - Found 8 tables to export
2026-03-24 15:09:40,698 - __main__ - INFO - Exporting customer to ./data/customer.csv
2026-03-24 15:09:40,779 - __main__ - INFO - Exported customer to ./data/customer.csv
2026-03-24 15:09:40,779 - __main__ - INFO - Exporting lineitem to ./data/lineitem.csv
2026-03-24 15:09:41,239 - __main__ - INFO - Exported lineitem to ./data/lineitem.csv
2026-03-24 15:09:41,239 - __main__ - INFO - Exporting nation to ./data/nation.csv
2026-03-24 15:09:41,255 - __main__ - INFO - Exported nation

In [14]:
%%sql --show
use prod_db

""


In [15]:
%%sql
SELECT
    l_orderkey,
    round( sum(l_extendedprice * (1 - l_discount) * (1 + l_tax)),
        2
    ) AS totalprice
FROM
    lineitem
WHERE
    l_orderkey = 1
GROUP BY
    l_orderkey;


,l_orderkey,totalprice
0,1,194029.59


In [16]:
%%sql

-- The totalprice of an order (with orderkey = 1)
SELECT
    o_orderkey,
    o_totalprice
FROM
    orders
WHERE
    o_orderkey = 1;



,o_orderkey,o_totalprice
0,1,194029.55


In [17]:
%%sql
    CREATE OR REPLACE TABLE dim_customer AS 
 SELECT 
        c.*,
        n_name AS nation_name,
        n_comment AS nation_comment,
        r_name AS region_name,
        r_comment AS region_comment
    FROM customer c
    LEFT JOIN nation n ON c_nationkey = n_nationkey
    LEFT JOIN region r ON n_regionkey = r_regionkey

'Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `prod_db`.`dim_customer` because it already exists.\nChoose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07'

In [18]:
%%sql
CREATE OR REPLACE TABLE fct_orders AS select * from orders

'Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `prod_db`.`fct_orders` because it already exists.\nChoose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07'

In [19]:
%%sql
CREATE OR REPLACE TABLE fct_lineitem AS select * from lineitem

'Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `prod_db`.`fct_lineitem` because it already exists.\nChoose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07'

In [20]:
%%sql
CREATE OR REPLACE TABLE wide_orders AS 
 SELECT 
        f.*,
        d.*
    FROM fct_orders f
    LEFT JOIN dim_customer d ON f.o_custkey = d.c_custkey

'Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `prod_db`.`wide_orders` because it already exists.\nChoose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07'

In [21]:
%%sql
CREATE OR REPLACE TABLE wide_lineitem AS select * from fct_lineitem

'Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `prod_db`.`wide_lineitem` because it already exists.\nChoose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07'

In [22]:
%%sql
CREATE OR REPLACE TABLE order_lineitem_metrics AS 
SELECT 
        l_orderkey as order_key,
        COUNT(l_linenumber) AS num_lineitems
    FROM wide_lineitem
    GROUP BY l_orderkey

'Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `prod_db`.`order_lineitem_metrics` because it already exists.\nChoose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07'

In [23]:
%%sql
CREATE OR REPLACE TABLE customer_outreach_metrics AS 
SELECT 
        wo.c_custkey,
        wo.c_name AS customer_name,
        MIN(wo.o_totalprice) AS min_order_value,
        MAX(wo.o_totalprice) AS max_order_value,
        AVG(wo.o_totalprice) AS avg_order_value,
        AVG(olm.num_lineitems) AS avg_num_items_per_order
    FROM wide_orders wo
    LEFT JOIN order_lineitem_metrics olm ON wo.o_orderkey = olm.order_key
    GROUP BY wo.c_custkey, wo.c_name

'Error: [TABLE_OR_VIEW_ALREADY_EXISTS] Cannot create table or view `prod_db`.`customer_outreach_metrics` because it already exists.\nChoose a different name, drop or replace the existing object, or add the IF NOT EXISTS clause to tolerate pre-existing objects. SQLSTATE: 42P07'

In [24]:
%%sql --show
select *
from customer_outreach_metrics

,c_custkey,customer_name,min_order_value,max_order_value,avg_order_value,avg_num_items_per_order
0,13678,Customer#000013678,3643.59,447542.26,150829.488571,3.666667
1,12484,Customer#000012484,3871.10,261383.06,122749.683636,3.545455
2,7940,Customer#000007940,53746.57,320282.96,165889.345455,4.909091
3,4945,Customer#000004945,2205.73,276173.48,127629.438947,3.842105
4,13276,Customer#000013276,58779.29,300437.66,149012.655238,4.000000
...,...,...,...,...,...,...
995,11326,Customer#000011326,54734.25,337839.46,171229.530000,5.000000
996,10553,Customer#000010553,41593.74,378066.38,146551.840000,3.714286
997,6380,Customer#000006380,57676.61,188571.97,117654.280000,3.000000
998,1265,Customer#000001265,38886.52,299235.43,184031.156250,4.625000
